In [ ]:
# Libraries to install:
# pip install pyserial # for serial communication
# pip install sensirion-slf3s # for using Sensision SLF3S-006F flow sensor with SSC1-USB cable

import argparse
from sensirion_shdlc_driver import ShdlcSerialPort, ShdlcConnection
from sensirion_uart_scc1.drivers.scc1_slf3x import Scc1Slf3x
from sensirion_uart_scc1.drivers.slf_common import get_flow_unit_label
from sensirion_uart_scc1.scc1_shdlc_device import Scc1ShdlcDevice
from scipy import stats
import numpy as np
import serial
import time

# PID controller class for feedback control
class PIDController:
    def __init__(self, kp, ki, kd, max_error):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.last_error = 0.0
        self.integral = 0.0
        self.max_error = max_error
        self.active = True

    def update(self, input, setpoint):
        error = setpoint - input
        if error > 500:
            error = 500 + (error - 500) * 0.5
        elif error < -500:
            error = -500 + (error + 500) * 0.5
        print ("PID error = {:.2f}".format(error))
        if abs(error) > self.max_error:
            self.active = True
            print ("PID active")
        if not self.active:
            print ("PID inactive")
            return self.last_pid_output
        self.integral += error
        print ("Integral = {:.2f}".format(self.integral))
        derivative = error - self.last_error
        output = self.kp * error + self.ki * self.integral + self.kd * derivative
        print ("Plunger Position = {:.2f} mm".format(output))
        # if output < 0:
        #     raise ValueError("Pump will hit limit switch\n" + "Operation terminated")
        self.last_error = error
        if abs(error) < self.max_error: # stop PID loop if error is within tolerance
            self.active = False
            self.last_pid_output = output
            return self.last_pid_output
        else:
            return round(output, 3)
    
    def reset(self):
        # for performing hard reset
        self.last_error = 0.0
        self.integral = 0.0
        self.active = True
        self.last_pid_output = 0.0
    
    def reset_integral(self, output):
        self.last_pid_output = output
        self.integral = 0.0  # ← clear history completely



# Note: Check if serial ports are correctly named
def initialize_serial_ports():
    grbl_port = serial.Serial('COM7', 115200) #Com 3 for new pump
    time.sleep(2)
    return grbl_port

# Function to check if the response from the GRBL controller is 'ok'
# Do not change this function!
def response_check(grbl_port):
    response = ''
    while 'ok' not in response:
        response = grbl_port.readline().decode().strip()
    return response

def send_homing_command(grbl_port, plngr_start_position):
    grbl_port.write(b'$H\n')
    response_check(grbl_port)
    print("Homing complete")
    grbl_port.write(b'G92 X0 Y0 Z0\n')
    grbl_port.write('G1 Z{:.2f} F1000\n'.format(plngr_start_position).encode())
    response_check(grbl_port)
    response_check(grbl_port)

# Automated valve initialization
def set_automated_valve_to_flow_mode_startup(grbl_port):
    grbl_port.write(b'G1 X4.0 F2000\n')
    response_check(grbl_port)
    time.sleep(1.0)

# Hard flow reset function
def hard_flow_reset(grbl_port, plngr_start_position):
    grbl_port.write(b'G1 X0.0 F2000\n')
    response_check(grbl_port)
    grbl_port.write('G1 Z{:.2f} F2000\n'.format(plngr_start_position).encode())
    response_check(grbl_port)
    grbl_port.write(b'G1 X3.8\n')
    response_check(grbl_port)
    return True

# flow sensor reading function
def read_flow(sensor, flow_scale):
    try:
        remaining, lost, data = sensor.read_extended_buffer()
        for flow, _, _ in data:
            return flow / flow_scale
    except UnicodeDecodeError:
        return None
    return None


def send_pid_output_to_grbl(grbl_port, pid_output, flow, pid):
    gcode_command = 'G1 Z{:.2f} F900\n'.format(pid_output)
    grbl_port.write(gcode_command.encode())

    response = ''
    start_time = time.time()
    while 'ok' not in response:
        if time.time() - start_time > 1.0:
            raise TimeoutError("GRBL controller did not respond:(")
        response = grbl_port.readline().decode().strip()
    return response

#############################################
######### Waveform G-code goes here #########
#############################################
def execute_waveform_loop(grbl_port, pid_output):
    for i in range(60):
        #Square wave
        grbl_port.write('G1 Z{:.2f} F2000\n'.format(pid_output + 3.70).encode())
        response_check(grbl_port)
        grbl_port.write('G1 Z{:.2f} F10\n'.format(pid_output + 4.15).encode())
        response_check(grbl_port)
        grbl_port.write('G04 P2.0\n'.encode())
        response_check(grbl_port)
        grbl_port.write('G1 Z{:.2f} F2000\n'.format(pid_output+0.45).encode())
        response_check(grbl_port)
        grbl_port.write('G1 Z{:.2f} F1\n'.format(pid_output+0.08).encode())
        response_check(grbl_port)
        grbl_port.write('G04 P2.0\n'.encode())
        response_check(grbl_port)
        print('loop count = {}, Plunger Position = {:.3f} mm'.format(i, pid_output))

    return (pid_output)
        

def main():
    
    #PID parameters
    kp = 0.004 # do not change kp = 0.040 for 75 um channel, kp = 0.010 for 200 um channel
    ki = 0.034 # do not change ki = 0.140 for 75 um channel,ki = 0.050 for 200 um channel
    kd = 0.000 # do not change
    plngr_start_position = 0 # plunger neutral position in mm
    Flow_sensor_acquisition_interval = 100 #in ms (recommended <100)

    pid_setpoint = 5.1 # initial flow setpoint in ul/min, needed to start the flow
    max_error = 0.1 # max error tolerance in ul/min


    inactive_delay = 0.1 * 60  # minutes x 60, initial laminar perfusion

    pid = PIDController(kp, ki, kd, max_error)

    grbl_port = initialize_serial_ports()

    with grbl_port:

        send_homing_command(grbl_port, plngr_start_position)
        set_automated_valve_to_flow_mode_startup(grbl_port)
        print("Homing cycle completed")
        parser = argparse.ArgumentParser()
        parser.add_argument('--serial-port', '-p', default='COM4')
        args, unknown = parser.parse_known_args()
        
        with ShdlcSerialPort(port=args.serial_port, baudrate=115200) as port:

            device = Scc1ShdlcDevice(ShdlcConnection(port), slave_address=0)
            device.set_sensor_type(Scc1Slf3x.SENSOR_TYPE)
            sensor = Scc1Slf3x(device)
            flow_scale, unit = sensor.get_flow_unit_and_scale()
            sensor.start_continuous_measurement(interval_ms=Flow_sensor_acquisition_interval)
            start_time = time.time()

            try:
                while True:

                    current_time = time.time()
                    elapsed_time = current_time - start_time # run time of the pump


                    flow = read_flow(sensor, flow_scale)
                    # if unable to read flow properly, skip the rest of the loop and retry
                    if flow is None:
                        continue
                    
                    #print("Did this reach here before flow")
                    pid_output = pid.update(flow, pid_setpoint)+ plngr_start_position

                    formatted_time = time.strftime("%m/%d/%Y %H:%M:%S", time.localtime(current_time))
                    print(formatted_time)

                    print("flow = {:.2f} ul/min, Plunger Position = {:.3f}".format(flow, pid_output))

                    if pid.active and elapsed_time>10 and ((pid_output+plngr_start_position)<0.2 or (pid_output+plngr_start_position)>24):
                        print("Plunger Position ({:.3f} mm) exceeded soft-limit, resetting pump".format(pid_output))
                        # Reset the pump if plunger translation exceeds 18 mm
                        pid_output = 0.0
                        pid.reset_integral(pid_output)
                        hard_flow_reset(grbl_port, plngr_start_position)
                        print("Reset successful")
                        start_time = time.time()
                        time.sleep(0.5)
                        continue

                    send_pid_output_to_grbl(grbl_port, pid_output, flow, pid)

                    if not pid.active and elapsed_time >= inactive_delay: 
                        # if code running for more than the set inactive_delay minutes and pid is inactive
                        pid_output = execute_waveform_loop(grbl_port, pid_output)
                        pid.reset_integral(pid_output)
                        time.sleep(8)
                    time.sleep(0.5)

                    if not pid.active and elapsed_time >= inactive_delay: 
                        time.sleep(0.5)

            except (serial.SerialException, TimeoutError, ValueError) as e:
                print("Error: ", e)
            
            finally:
                sensor.stop_continuous_measurement()
                grbl_port.write(b'G1 X1.7 F2000\n')
                response_check(grbl_port)
                pass

if __name__ == "__main__":
    main()